# Manual Classification Metrics
Evaluates the algorithmic `valid_flag` against manual labels.

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
MANUAL_LABELS_PATH = "../../results/summary/manual_labels.csv"

df = pd.read_csv(MANUAL_LABELS_PATH)
df = df[df["manual_label"] != "skip"].copy()

print(f"Total samples (excl. skip): {len(df)}")
print()
print("Manual label distribution:")
print(df["manual_label"].value_counts())
print()
print("Algo label distribution:")
print(df["valid_flag"].value_counts())

In [ ]:
y_true = df["manual_label"]
y_pred = df["valid_flag"]
labels = sorted(set(y_true) | set(y_pred))

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
recall    = recall_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
f1        = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)

print(f"Accuracy:             {accuracy:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall    (weighted): {recall:.4f}")
print(f"F1-score  (weighted): {f1:.4f}")

In [ ]:
print("Per-class report (manual = truth, algo = prediction):")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_true, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix\n(rows = manual, cols = algo)")
plt.tight_layout()
plt.show()

In [ ]:
report_df = pd.DataFrame(
    classification_report(y_true, y_pred, labels=labels, zero_division=0, output_dict=True)
).T.drop(index=["accuracy"], errors="ignore")

per_class = report_df.loc[labels, ["precision", "recall", "f1-score"]]

fig, ax = plt.subplots(figsize=(9, 4))
per_class.plot(kind="bar", ax=ax, width=0.7)
ax.set_title("Per-class Precision / Recall / F1")
ax.set_xlabel("Class")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(loc="lower right")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()